In [1]:
#Importing necessary libraries for data handling, modeling, handling class imbalance, and evaluating.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_curve
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

#Loading the training and testing datasets.
X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")
y_train = pd.read_csv("data/y_train.csv")
y_test = pd.read_csv("data/y_test.csv")

#Converting y_train and y_test to 1D arrays for modeling.
y_train = y_train.values.ravel()
y_test = y_test.values.ravel()

In [2]:
#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

#Printing the action classes and the number of classes.
print(actions)
print(f'The number of unique classes is {len(actions)}.')

['appear' 'click' 'disappear' 'drag' 'hover' 'scrolldown' 'scrollup'
 'select' 'type' 'zoomin' 'zoomout']
The number of unique classes is 11.


In [3]:
#Defining the rare classes.
rare_classes = ["type", "hover", "scrolldown", "appear", "disappear", "scrollup"]

#Replacing rare classes with "other" in the training labels.
y_train = pd.Series(y_train).replace(rare_classes, "other").values

#Replacing rare classes with "other" in the testing labels.
y_test = pd.Series(y_test).replace(rare_classes, "other").values

#Getting the unique action classes from the training labels.
actions = np.sort(np.unique(y_train))

In [4]:
#Storing the best hyperparameters found from GridSearchCV for each class.
best_params = {"click": {"C": 10.0, "l1_ratio": 0.7}, "drag": {"C": 1.0,  "l1_ratio": 0.7},
"other": {"C": 10.0, "l1_ratio": 0.3}, "select": {"C": 10.0, "l1_ratio": 0.3},
"zoomin": {"C": 10.0, "l1_ratio": 0.3},"zoomout": {"C": 10.0, "l1_ratio": 0.5}}

#Splitting off a validation set from training data to use for threshold tuning.
X_train_fold, X_val, y_train_fold, y_val = train_test_split(X_train, y_train, test_size = 0.2, random_state = 42, stratify = y_train)

#Storing the trained models, scalers, thresholds, and evaluation metrics for each class.
ova_models = {}
ova_scalers = {}
ova_thresholds = {}
ova_results = {}

#Looping through each action class to build the OvA models.
for action in actions:

    #Creating binary labels for the training fold, validation set, and test set.
    y_train_binary = (y_train_fold == action).astype(int)
    y_val_binary   = (y_val == action).astype(int)
    y_test_binary  = (y_test == action).astype(int)

    #Applying SMOTE to the training fold to balance the binary classes.
    smote = SMOTE(random_state = 42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fold, y_train_binary)

    #Scaling the resampled training data and applying the same scaler to validation and test.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_resampled)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    #Training an Elastic Net logistic regression model with increased max_iter.
    model = LogisticRegression(penalty = "elasticnet", solver = "saga", C = best_params[action]["C"],
        l1_ratio = best_params[action]["l1_ratio"], max_iter = 7200, random_state = 42)
    model.fit(X_train_scaled, y_train_resampled)

    #Tuning the classification threshold on the validation set (FIX 1).
    y_val_probs = model.predict_proba(X_val_scaled)[:, 1]
    precision_vals, recall_vals, thresholds = precision_recall_curve(y_val_binary, y_val_probs)
    f1_scores = 2 * (precision_vals[:-1] * recall_vals[:-1]) / (precision_vals[:-1] + recall_vals[:-1] + 1e-8)
    best_threshold = thresholds[np.argmax(f1_scores)]

    #Storing the trained model, scaler, and threshold for the current class.
    ova_models[action] = model
    ova_scalers[action] = scaler
    ova_thresholds[action] = best_threshold

    #Predicting probabilities on the test set and applying the tuned threshold.
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    y_pred = (y_prob >= best_threshold).astype(int)

    #Calculating evaluation metrics for the current class.
    accuracy = accuracy_score(y_test_binary, y_pred)
    precision = precision_score(y_test_binary, y_pred, average = "macro", zero_division = 0)
    recall = recall_score(y_test_binary, y_pred, average = "macro", zero_division = 0)
    f1 = f1_score(y_test_binary, y_pred, average = "macro", zero_division = 0)

    #Storing the evaluation metrics and the tuned threshold for the current class.
    ova_results[action] = {"Accuracy": accuracy, "Precision": precision, "Recall": recall,
    "F1": f1, "Threshold": best_threshold}

#Converting the stored results into a DataFrame.
results_df = pd.DataFrame(ova_results).T
print("Per-class binary results:")
print(results_df)

Per-class binary results:
         Accuracy  Precision    Recall        F1  Threshold
click    0.680045   0.671668  0.695323  0.666620   0.527945
drag     0.702341   0.711510  0.706805  0.701486   0.411805
other    0.947603   0.577772  0.655616  0.601194   0.866203
select   0.949833   0.604332  0.677918  0.629859   0.864530
zoomin   0.968785   0.908576  0.865383  0.885516   0.906796
zoomout  0.963211   0.868559  0.863496  0.866006   0.815160


In [5]:
#Combining the 6 OvA binary classifiers into a single multi-class prediction.

#Collecting the predicted probability of being the positive class from each model.
prob_matrix = pd.DataFrame(index = range(len(X_test)), columns = actions, dtype = float)

for action in actions:
    scaler = ova_scalers[action]
    model = ova_models[action]
    X_test_scaled = scaler.transform(X_test)
    prob_matrix[action] = model.predict_proba(X_test_scaled)[:, 1]

#Assigning each sample the class with the highest predicted probability.
y_pred_multiclass = prob_matrix.idxmax(axis = 1).values

#Evaluating the combined multi-class predictions.
multiclass_accuracy = accuracy_score(y_test, y_pred_multiclass)
multiclass_precision = precision_score(y_test, y_pred_multiclass, average = "macro", zero_division = 0)
multiclass_recall = recall_score(y_test, y_pred_multiclass, average = "macro", zero_division = 0)
multiclass_f1 = f1_score(y_test, y_pred_multiclass, average = "macro", zero_division = 0)

print("Combined OvA Multi-class Results:")
print(f"Accuracy: {multiclass_accuracy:.4f}")
print(f"Precision: {multiclass_precision:.4f} (macro)")
print(f"Recall: {multiclass_recall:.4f} (macro)")
print(f"F1: {multiclass_f1:.4f} (macro)")

#Per-class breakdown of multi-class predictions.
from sklearn.metrics import classification_report
print("\nPer-class classification report:")
print(classification_report(y_test, y_pred_multiclass, zero_division = 0))

Combined OvA Multi-class Results:
Accuracy: 0.5596
Precision: 0.4783 (macro)
Recall: 0.5975 (macro)
F1: 0.4969 (macro)

Per-class classification report:
              precision    recall  f1-score   support

       click       0.55      0.49      0.52       291
        drag       0.75      0.54      0.63       426
       other       0.14      0.50      0.22        20
      select       0.12      0.48      0.19        23
      zoomin       0.61      0.84      0.71        70
     zoomout       0.70      0.73      0.72        67

    accuracy                           0.56       897
   macro avg       0.48      0.60      0.50       897
weighted avg       0.64      0.56      0.58       897

